# Locate Ameriflux Station & Download Forcing

This notebook helps you:
1. Find an Ameriflux flux tower site using HydroData
2. Query CONUS2 subsurface parameters (soil type, water table depth)
3. Prepare CLM input files (vegm, clmin)
4. Download CW3E meteorological forcing

**Prerequisites:** HydroData account (free) — see README.md for signup instructions.

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import subsettools as st
import hf_hydrodata as hf
import shutil
from pathlib import Path

from helpers import IGBP_NAMES, IGBP_STR_TO_INT, CONUS2_SOILS, build_vegm, patch_clmin_dates

# Register your HydroData PIN (only needed once per machine)
# Sign up: https://hydrogen.princeton.edu/signup
# Get PIN: https://hydrogen.princeton.edu/pin
hf.register_api_pin("your_email@example.com", "your_pin")

In [ ]:
# ============================================================
# User Configuration
# ============================================================
# Choose a site and water year. These are used throughout all 3 notebooks.

site_id = "US-Slt"          # Ameriflux site ID
start   = "2011-10-01"      # Water year start (Oct 1)
end     = "2012-10-01"      # Water year end (Oct 1)
water_year = 2012

# Directory where forcing and CLM input files will be written
# This is also where you will run ParFlow in Notebook 2
static_write_dir = "/Users/reed/Projects/single_column_PFCLM/runs/US_Slt_WY2012"

# -----------------------------------------------------------
# Tested sites from the ET sensitivity study for reference:
#   US-Slt  (DBF, NJ,  WY2012)   US-xUK  (DBF, KS,  WY2024)
#   US-xBL  (DBF, VA,  WY2024)   US-GLE  (ENF, WY,  WY2020)
#   US-Ro5  (CRO, MN,  WY2024)   US-Ro1  (CRO, MN,  WY2016)
#   US-Ro2  (CRO, MN,  WY2016)   US-Fwf  (GRA, AZ,  WY2008)
#   US-Mpj  (WSA, NM,  WY2024)   US-Ton  (WSA, CA,  WY2024)
#   US-xSR  (OSH, AZ,  WY2024)   US-xJR  (OSH, NM,  WY2024)
#   US-xDL  (MF,  AL,  WY2024)
# -----------------------------------------------------------

In [ ]:
# Query Ameriflux metadata by state
State = "NJ"  # Change to search other states

metadata_df = hf.get_point_metadata(
    dataset="ameriflux",
    variable="latent_heat",
    temporal_resolution="hourly",
    aggregation="sum",
    state=State,
)
print(f"Found {len(metadata_df)} Ameriflux sites in {State}:")
metadata_df[["site_id", "site_name", "latitude", "longitude",
             "first_date_data_available", "last_date_data_available"]]

In [ ]:
# Interactive map of sites (Bokeh)
from bokeh.plotting import figure, output_notebook, show
from bokeh.models import ColumnDataSource, LabelSet

output_notebook()

def wgs84_to_web_mercator(lon, lat):
    k = 6378137
    x = lon * (k * np.pi / 180.0)
    y = np.log(np.tan((90 + lat) * np.pi / 360.0)) * k
    return x, y

metadata_df["x"] = metadata_df.apply(
    lambda r: wgs84_to_web_mercator(r["longitude"], r["latitude"])[0], axis=1)
metadata_df["y"] = metadata_df.apply(
    lambda r: wgs84_to_web_mercator(r["longitude"], r["latitude"])[1], axis=1)

source = ColumnDataSource(data=dict(
    x=list(metadata_df["x"]), y=list(metadata_df["y"]),
    site_id=list(metadata_df["site_id"]),
    site_name=list(metadata_df["site_name"]),
))

buf = 0.2
x_min, y_min = wgs84_to_web_mercator(metadata_df["longitude"].min() - buf,
                                       metadata_df["latitude"].min() - buf)
x_max, y_max = wgs84_to_web_mercator(metadata_df["longitude"].max() + buf,
                                       metadata_df["latitude"].max() + buf)

p = figure(x_range=(x_min, x_max), y_range=(y_min, y_max),
           x_axis_type="mercator", y_axis_type="mercator",
           title="Ameriflux Sites")
p.add_tile("OSM")
p.scatter(x="x", y="y", size=15, fill_color="red", fill_alpha=0.8, source=source)
labels = LabelSet(x="x", y="y", text="site_id", level="glyph",
                  text_font_size="9pt", x_offset=5, y_offset=5, source=source)
p.add_layout(labels)
show(p)

In [ ]:
# Extract site info from metadata
site_info = metadata_df[metadata_df["site_id"] == site_id]
if site_info.empty:
    raise ValueError(f"Site {site_id} not found in metadata. Check state or search by lat/lon.")

lat = site_info["latitude"].values[0]
lon = site_info["longitude"].values[0]

# Get IGBP type from Ameriflux metadata
igbp_str = site_info["ameriflux_igbp"].values[0]
igbp_type = IGBP_STR_TO_INT.get(igbp_str, None)
print(f"Site: {site_id}")
print(f"  lat={lat:.4f}, lon={lon:.4f}")
print(f"  IGBP: {igbp_str} (index={igbp_type})")
print(f"  Data available: {site_info['first_date_data_available'].values[0]} to "
      f"{site_info['last_date_data_available'].values[0]}")

In [ ]:
# Preview Ameriflux latent heat data with Bokeh range tool
from bokeh.models import RangeTool
from bokeh.layouts import column as bk_column

lh_df = hf.get_point_data(
    dataset="ameriflux", variable="latent_heat",
    temporal_resolution="hourly", aggregation="sum",
    date_start=start, date_end=end, site_ids=site_id,
)
lh_dates = pd.to_datetime(lh_df["date"])
lh_vals = lh_df[site_id].values.astype(float)
lh_vals[lh_vals == -9999] = np.nan

source_lh = ColumnDataSource(data=dict(x=lh_dates, y=lh_vals))

p_main = figure(height=300, width=800, title=f"{site_id} Latent Heat (Ameriflux)",
                x_axis_type="datetime", y_axis_label="LH [W/m2]",
                x_range=(lh_dates.iloc[0], lh_dates.iloc[-1]))
p_main.line("x", "y", source=source_lh, color="black", line_width=0.5)

select = figure(height=130, width=800, x_axis_type="datetime",
                y_range=p_main.y_range, title="Drag to select range")
select.line("x", "y", source=source_lh, color="black", line_width=0.3)
range_tool = RangeTool(x_range=p_main.x_range)
select.add_tools(range_tool)

show(bk_column(p_main, select))

## CONUS2 Grid & Subsurface Parameters

Query the CONUS2 domain for soil type, water table depth (WTD), and sand/clay fractions at the site location. These inform the ParFlow soil parameters in Notebook 2.

In [ ]:
# Define single-column CONUS2 domain
latlon_bounds = [[lat - 0.005, lon - 0.005], [lat + 0.005, lon + 0.005]]
ij_bounds, mask = st.define_latlon_domain(latlon_bounds=latlon_bounds, grid="conus2")
print(f"CONUS2 grid bounds (i_start, j_start, i_end, j_end): {ij_bounds}")

# Query soil type from CONUS2 pf_indicator
ind = hf.get_gridded_data({
    "dataset": "conus2_domain", "variable": "pf_indicator",
    "grid_bounds": ij_bounds,
})
soil_type = int(ind[-1, 0, 0])  # top layer
soil_info = CONUS2_SOILS[soil_type]
print(f"\nCONUS2 soil type: {soil_type} ({soil_info[0]})")
print(f"  Ksat={soil_info[1]} m/h, porosity={soil_info[2]}, "
      f"VG alpha={soil_info[3]}, VG n={soil_info[4]}, Sres={soil_info[5]}")

# Query CONUS2 baseline WTD (monthly mean)
try:
    wtd_c2 = hf.get_gridded_data({
        "dataset": "conus2_baseline", "variable": "water_table_depth",
        "period": "daily", "start_time": start, "end_time": end,
        "grid_bounds": ij_bounds,
    })
    wtd_c2_mean = float(np.nanmean(wtd_c2[:, 0, 0]))
    print(f"\nCONUS2 baseline mean WTD: {wtd_c2_mean:.2f} m")
except Exception as e:
    wtd_c2_mean = None
    print(f"\nCONUS2 baseline WTD unavailable: {e}")

# Query Ma et al. 2025 30m hi-res WTD
try:
    i_s, j_s = ij_bounds[0], ij_bounds[1]
    wtd_30m = hf.get_gridded_data({
        "dataset": "ma_2025", "variable": "water_table_depth",
        "grid": "conus2_wtd.30", "grid_bounds": (i_s*33, j_s*33, i_s*33+1, j_s*33+1),
    })
    wtd_30m_val = float(wtd_30m.flatten()[0])
    print(f"Ma et al. 2025 30m WTD: {wtd_30m_val:.2f} m")
except Exception as e:
    wtd_30m_val = None
    print(f"Ma 2025 WTD unavailable: {e}")

print(f"\n--- Summary ---")
print(f"CONUS2 baseline WTD: {wtd_c2_mean}")
print(f"Ma 2025 30m WTD:     {wtd_30m_val}")
print(f"Pick a value below (negative = below surface for ParFlow BC).")

In [ ]:
# ============================================================
# Vegm parameters — edit these if needed
# ============================================================
# Sand/clay: try to get from subsettools config_clm, fall back to defaults
try:
    _clm_paths = st.config_clm(ij_bounds, start=start, end=end,
                                dataset="conus2_domain", write_dir="/tmp/_clm_tmp")
    # Read sand/clay from the generated vegm
    with open("/tmp/_clm_tmp/drv_vegm.dat") as _f:
        _lines = _f.readlines()
    for _line in _lines:
        _parts = _line.split()
        if len(_parts) >= 25:
            sand_frac = float(_parts[4])
            clay_frac = float(_parts[5])
            color_index = int(_parts[6])
            break
    print(f"From CONUS2: sand={sand_frac}, clay={clay_frac}, color={color_index}")
except Exception as e:
    print(f"config_clm failed ({e}), using defaults")
    sand_frac = 0.40
    clay_frac = 0.20
    color_index = 4

# Override IGBP if needed (e.g. US-Slt is ENF in reality but CONUS2 says DBF)
# igbp_type = 1  # uncomment to override

print(f"\nVegm parameters:")
print(f"  sand={sand_frac}, clay={clay_frac}, color={color_index}")
print(f"  IGBP={igbp_type} ({IGBP_NAMES.get(igbp_type, '?')})")

## Prepare Run Directory

Write CLM input files (drv_vegm.dat, drv_clmin.dat, drv_vegp.dat) to the run directory.

In [ ]:
# Create run directory and write CLM input files
run_dir = Path(static_write_dir)
run_dir.mkdir(parents=True, exist_ok=True)

# 1. Write drv_vegm.dat
build_vegm(run_dir / "drv_vegm.dat", lat, lon, sand_frac, clay_frac, color_index, igbp_type)
print(f"Wrote drv_vegm.dat (IGBP={igbp_type})")

# 2. Patch drv_clmin.dat dates from template
clm_inputs = Path("clm_inputs")
patch_clmin_dates(clm_inputs / "drv_clmin_template.dat",
                  run_dir / "drv_clmin.dat", water_year)
print(f"Wrote drv_clmin.dat (WY{water_year})")

# 3. Copy drv_vegp.dat (shipped with improved PFT parameters + clumping)
shutil.copy2(clm_inputs / "drv_vegp.dat", run_dir / "drv_vegp.dat")
print(f"Copied drv_vegp.dat (CLM4.5 + canopy clumping)")

print(f"\nRun directory: {run_dir}")
print("Contents:", [f.name for f in sorted(run_dir.iterdir())])

## Download CW3E Forcing

Download 8 meteorological forcing variables from the CW3E reanalysis dataset via HydroData. This can take a few minutes.

In [ ]:
# CW3E forcing variables in ParFlow-CLM 1D forcing order:
#   DSWR  - Downward shortwave radiation [W/m2]
#   DLWR  - Downward longwave radiation [W/m2]
#   APCP  - Precipitation rate [mm/s]
#   Temp  - Air temperature [K]
#   UGRD  - East-west wind speed [m/s]
#   VGRD  - North-south wind speed [m/s]
#   Press - Atmospheric pressure [Pa]
#   SPFH  - Specific humidity [kg/kg]

forcing_vars = [
    "downward_shortwave", "downward_longwave", "precipitation",
    "air_temp", "east_windspeed", "north_windspeed",
    "atmospheric_pressure", "specific_humidity",
]

# Calculate expected hours
from datetime import datetime
dt_start = datetime.strptime(start, "%Y-%m-%d")
dt_end = datetime.strptime(end, "%Y-%m-%d")
n_hours = int((dt_end - dt_start).total_seconds() / 3600)
print(f"Downloading {n_hours} hours of forcing for {site_id}...")

forcing_data = np.zeros((n_hours, 8))

for i, var in enumerate(forcing_vars):
    print(f"  [{i+1}/8] {var}...", end=" ", flush=True)
    options = {
        "dataset": "CW3E", "period": "hourly", "variable": var,
        "start_time": start, "end_time": end,
        "grid_bounds": ij_bounds, "dataset_version": "1.0",
    }
    data = hf.get_gridded_data(options)
    col = data[:, 0, 0] if data.ndim == 3 else data.flatten()
    if len(col) != n_hours:
        print(f"WARNING: got {len(col)} hours, expected {n_hours}")
        col = col[:n_hours] if len(col) > n_hours else np.pad(col, (0, n_hours - len(col)))
    forcing_data[:, i] = col
    print(f"OK (range: {col.min():.3g} to {col.max():.3g})")

# Write forcing file
forcing_filename = f"forcing1D.{site_id}.{start}-{end}.txt"
forcing_path = run_dir / forcing_filename
np.savetxt(forcing_path, forcing_data, fmt="%.6e", delimiter=" ")
print(f"\nSaved: {forcing_path} ({forcing_data.shape[0]} hours, "
      f"{forcing_path.stat().st_size/1024:.0f} KB)")

In [ ]:
# Plot forcing: precipitation and temperature time series
dates = pd.date_range(start, periods=n_hours, freq="h")

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

axes[0].plot(dates, forcing_data[:, 2] * 3600, color="steelblue", lw=0.5)
axes[0].set_ylabel("Precip [mm/h]")
axes[0].set_title(f"{site_id} CW3E Forcing (WY{water_year})")

axes[1].plot(dates, forcing_data[:, 3] - 273.15, color="firebrick", lw=0.3)
axes[1].set_ylabel("Air Temp [C]")
axes[1].set_xlabel("Date")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))

plt.tight_layout()
plt.show()

## Summary

Your run directory is ready with:
- `drv_vegm.dat` — land cover and soil texture
- `drv_clmin.dat` — CLM driver with correct dates
- `drv_vegp.dat` — PFT parameters (CLM4.5 + canopy clumping)
- `forcing1D.*.txt` — CW3E meteorological forcing

**Key parameters for Notebook 2:**
- `soil_type` — CONUS2 soil indicator (for hydraulic properties lookup)
- `water_year`, `start`, `end` — simulation dates
- `static_write_dir` — run directory path

Proceed to **Single_Column_PFCLM_netcdf.ipynb** to configure and run the simulation.